# MLOps Assignment 2 — M1: Model Development & Experiment Tracking
**Use case:** Binary image classification (Cats vs Dogs) for a pet adoption platform.

This notebook covers:
1. Data & Code Versioning (Git + DVC)
2. Model Building (baseline CNN, PyTorch)
3. Experiment Tracking (MLflow — params, metrics, confusion matrix, loss curves)

> Run cells top to bottom. Some cells are meant to be run once (dataset download/split, git/dvc init) — they're marked accordingly.

## 0. Environment Setup
Install dependencies (run once).

In [1]:
# Run once per environment
!pip install kagglehub dvc mlflow scikit-learn matplotlib torch torchvision pillow -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [2]:
import os
import shutil
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import mlflow
import mlflow.pytorch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.13.0+cpu
CUDA available: False


In [16]:
import os

# Move working directory from notebooks/ up to project root — run once per kernel session
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working directory:", os.getcwd())

Working directory: C:\Users\XRIG\cats-dogs-mlops


## 1. Data & Code Versioning

### 1.1 Download the dataset (KaggleHub)
Downloads the Cats vs Dogs dataset locally. Requires a Kaggle account/API token configured
(`~/.kaggle/kaggle.json`) if kagglehub prompts for auth.

In [17]:
import kagglehub

path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")
print("Dataset downloaded to:", path)

for root, dirs, files in os.walk(path):
    print(root, "->", dirs[:5], f"({len(files)} files)")
    if root.count(os.sep) - path.count(os.sep) > 2:
        break

Dataset downloaded to: C:\Users\XRIG\.cache\kagglehub\datasets\shaunthesheep\microsoft-catsvsdogs-dataset\versions\1
C:\Users\XRIG\.cache\kagglehub\datasets\shaunthesheep\microsoft-catsvsdogs-dataset\versions\1 -> ['PetImages'] (2 files)
C:\Users\XRIG\.cache\kagglehub\datasets\shaunthesheep\microsoft-catsvsdogs-dataset\versions\1\PetImages -> ['Cat', 'Dog'] (0 files)
C:\Users\XRIG\.cache\kagglehub\datasets\shaunthesheep\microsoft-catsvsdogs-dataset\versions\1\PetImages\Cat -> [] (12500 files)
C:\Users\XRIG\.cache\kagglehub\datasets\shaunthesheep\microsoft-catsvsdogs-dataset\versions\1\PetImages\Dog -> [] (12500 files)


### 1.2 Preprocessing functions

These live logically in `src/data_preprocessing.py` for reuse by the FastAPI service (M2) and
unit tests (M3). Defined here inline so the notebook is fully self-contained and runnable
end-to-end; copy this cell's contents into `src/data_preprocessing.py` for the project repo.

In [7]:
IMAGE_SIZE = (224, 224)
SPLIT_RATIOS = {"train": 0.8, "val": 0.1, "test": 0.1}

def list_class_images(class_dir: str) -> list:
    """Return sorted list of valid image file paths in a class directory."""
    valid_ext = {".jpg", ".jpeg", ".png"}
    return sorted([
        str(Path(class_dir) / f)
        for f in os.listdir(class_dir)
        if Path(f).suffix.lower() in valid_ext
    ])

def split_dataset(file_list: list, ratios: dict = SPLIT_RATIOS, seed: int = 42) -> dict:
    """Deterministically split a file list into train/val/test given ratios."""
    random.Random(seed).shuffle(file_list)
    n = len(file_list)
    n_train = int(n * ratios["train"])
    n_val = int(n * ratios["val"])
    return {
        "train": file_list[:n_train],
        "val": file_list[n_train:n_train + n_val],
        "test": file_list[n_train + n_val:],
    }

def build_split_dirs(source_root: str, dest_root: str, classes: list = ("cats", "dogs")):
    """Reorganize a flat/class-folder dataset into data/raw/{train,val,test}/{class}/."""
    for split in ["train", "val", "test"]:
        for cls in classes:
            os.makedirs(Path(dest_root) / split / cls, exist_ok=True)

    for cls in classes:
        class_dir = Path(source_root) / cls
        files = list_class_images(str(class_dir))
        splits = split_dataset(files)
        for split_name, split_files in splits.items():
            for f in split_files:
                shutil.copy(f, Path(dest_root) / split_name / cls / Path(f).name)

    return {s: sum(len(os.listdir(Path(dest_root)/s/c)) for c in classes) for s in ["train", "val", "test"]}

In [8]:
source_root = f"{path}/PetImages"
counts = build_split_dirs(source_root=source_root, dest_root="data/raw", classes=("Cat", "Dog"))

### 1.3 Filter corrupt images (optional but recommended)
The public Cats vs Dogs dataset has a handful of malformed/0-byte JPEGs. Run this before
splitting if you hit `PIL.UnidentifiedImageError` later.

In [10]:
from PIL import Image

def filter_corrupt_images(class_dir: str):
    removed = []
    for f in list_class_images(class_dir):
        try:
            img = Image.open(f)
            img.verify()
        except Exception:
            removed.append(f)
            os.remove(f)
    return removed

# Example usage — adjust to your extracted path structure:
source_root = f"{path}/PetImages"
for cls in ["Cat", "Dog"]:
    bad = filter_corrupt_images(f"{source_root}/{cls}")
    print(f"{cls}: removed {len(bad)} corrupt files")

Cat: removed 1 corrupt files
Dog: removed 1 corrupt files


C:\Users\XRIG\AppData\Roaming\Python\Python313\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


### 1.4 Run the split (adjust `source_root` to match your extracted folder names)

In [14]:
def build_split_dirs(source_root: str, dest_root: str, source_classes=("Cat", "Dog"), dest_classes=("cats", "dogs")):
    for split in ["train", "val", "test"]:
        for cls in dest_classes:
            os.makedirs(Path(dest_root) / split / cls, exist_ok=True)

    for src_cls, dst_cls in zip(source_classes, dest_classes):
        class_dir = Path(source_root) / src_cls
        files = list_class_images(str(class_dir))
        splits = split_dataset(files)
        for split_name, split_files in splits.items():
            for f in split_files:
                shutil.copy(f, Path(dest_root) / split_name / dst_cls / Path(f).name)

    return {s: sum(len(os.listdir(Path(dest_root)/s/c)) for c in dest_classes) for s in ["train", "val", "test"]}

In [15]:
counts = build_split_dirs(source_root=f"{path}/PetImages", dest_root="data/raw")
print("Split counts:", counts)

Split counts: {'train': 19998, 'val': 2498, 'test': 2502}


### 1.5 Git init + .gitignore (run once, from a terminal or via `!` shell commands)

```
git init
```

Create a `.gitignore`:
```
data/raw/
mlruns/
*.pyc
__pycache__/
.ipynb_checkpoints/
venv/
.env
models/*.pt
```

```
git add .
git commit -m "Initial project structure"
```

In [ ]:
# Optional: run git commands directly from the notebook (uncomment to use)
# !git init
# !git add .
# !git commit -m "Initial project structure"

### 1.6 DVC setup for dataset & model versioning (run once)

```
dvc init
git add .dvc .dvcignore
git commit -m "Initialize DVC"

dvc add data/raw
git add data/raw.dvc .gitignore
git commit -m "Track dataset v1 with DVC"

mkdir -p ../dvc-storage
dvc remote add -d local_storage ../dvc-storage
dvc push
git add .dvc/config
git commit -m "Configure local DVC remote"
```

In [ ]:
# Optional: run DVC commands directly from the notebook (uncomment to use)
# !dvc init
# !dvc add data/raw
# !git add data/raw.dvc .gitignore
# !git commit -m "Track dataset v1 with DVC"

## 2. Model Building

### 2.1 Baseline CNN architecture
Also lives logically in `src/model.py` for reuse by the inference service.

In [ ]:
class BaselineCNN(nn.Module):
    """Simple CNN baseline for binary cats-vs-dogs classification on 224x224 RGB images."""
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 224->112
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 112->56
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 56->28
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 28->14
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 14 * 14, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# Sanity check
_dummy = torch.randn(2, 3, 224, 224)
_model = BaselineCNN()
print("Output shape:", _model(_dummy).shape)  # expect torch.Size([2, 2])

### 2.2 Data loaders with augmentation

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder("data/raw/train", transform=train_tf)
val_ds   = datasets.ImageFolder("data/raw/val", transform=eval_tf)
test_ds  = datasets.ImageFolder("data/raw/test", transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

print("Class mapping:", train_ds.class_to_idx)  # note this — you'll need it in the API (M2)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

## 3. Experiment Tracking with MLflow

Logs hyperparameters, per-epoch metrics, a confusion matrix artifact, a loss-curve artifact,
and the trained model itself.

> In a separate terminal, run `mlflow ui --port 5000` and open `http://localhost:5000` to
> view runs while/after this cell executes.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = BaselineCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

mlflow.set_experiment("cats-vs-dogs-baseline")

EPOCHS = 10

In [ ]:
with mlflow.start_run(run_name="baseline_cnn_v1"):
    mlflow.log_params({
        "epochs": EPOCHS,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "architecture": "BaselineCNN",
        "image_size": 224,
    })

    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        train_loss = running_loss / len(train_ds)
        train_losses.append(train_loss)

        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                correct += (outputs.argmax(1) == labels).sum().item()
        val_loss /= len(val_ds)
        val_acc = correct / len(val_ds)
        val_losses.append(val_loss)

        mlflow.log_metrics({
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        }, step=epoch)

        print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    # --- Final test evaluation ---
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
    mlflow.log_metric("test_accuracy", test_acc)

    # --- Confusion matrix artifact ---
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay(cm, display_labels=["cats", "dogs"]).plot(ax=ax, cmap="Blues")
    plt.title("Test Confusion Matrix")
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.show()
    plt.close(fig)

    # --- Loss curve artifact ---
    fig, ax = plt.subplots()
    ax.plot(train_losses, label="train_loss")
    ax.plot(val_losses, label="val_loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()
    plt.title("Loss Curves")
    plt.savefig("loss_curves.png")
    mlflow.log_artifact("loss_curves.png")
    plt.show()
    plt.close(fig)

    # --- Save & log model ---
    os.makedirs("models", exist_ok=True)
    torch.save(model.state_dict(), "models/baseline_cnn.pt")
    mlflow.log_artifact("models/baseline_cnn.pt")
    mlflow.pytorch.log_model(model, "pytorch_model")

    print(f"\nFinal test accuracy: {test_acc:.4f}")

### 3.1 Commit the trained model artifact with DVC (run once, from terminal)

```
dvc add models/baseline_cnn.pt
git add models/baseline_cnn.pt.dvc mlruns/.gitignore src/ notebooks/
git commit -m "M1: baseline CNN trained, tracked with MLflow and DVC"
dvc push
```

In [ ]:
# Optional: uncomment to run from the notebook
# !dvc add models/baseline_cnn.pt
# !git add models/baseline_cnn.pt.dvc
# !git commit -m "M1: baseline CNN trained, tracked with MLflow and DVC"
# !dvc push

## Summary — M1 checklist
- [x] Git-tracked source code (this notebook + `src/` modules)
- [x] DVC-tracked dataset (`data/raw`) and model artifact (`models/baseline_cnn.pt`)
- [x] Baseline CNN trained and saved in `.pt` format
- [x] MLflow run logged with params, per-epoch metrics, confusion matrix, and loss-curve artifacts

Next: **M2 — Model Packaging & Containerization** (FastAPI inference service + Dockerfile).